# GraceDB BNS/NSBH SESN 3D Crossmatch (Refactor)

*This is a refactor of the original gracedb_sesn_3d_crossmatch.ipynb notebook. It replaces constants and functions previously define within the notebook with calls to modules that do the same.*

This notebook downloads public GraceDB production superevents passing the configured FAR threshold, keeps superevents passing the configured BNS/NSBH probability cut, downloads the best available 3D sky map, and crossmatches stripped-envelope supernovae from TNS.

The workflow is:

1. Download and filter the TNS public catalog for SESN-like types near the configured distance limit.
2. Query GraceDB superevents, fetch `p_astro.json` classifications, and download one best available multiorder FITS skymap per passing superevent.
3. Reproduce the temporal match from `how_many_SESN.ipynb` using the configured discovery-time window.
4. Run a configurable 3D credible-volume crossmatch twice: once with SN luminosity distances from the SHOES cosmology and once with `Planck18`.
5. If any SN lands inside the configured 3D credible volume, save diagnostic overlap plots to `gracedb_sesn_3d_plots/`.


In [ ]:
# For development, automatically reload modules when they change

%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

## Get transients from TNS

In [ ]:
from desi_aap.tns_catalog import download_tns_table, clean_tns_catalog

In [ ]:
tns_raw = download_tns_table()
df_sesn = clean_tns_catalog(tns_raw)

df_sesn

In [ ]:
col_name = "dist_mpc_Planck18"  # SHOES"

min_dist = df_sesn[col_name].min()
max_dist = df_sesn[col_name].max()

print("Minimum distance:", min_dist)
print("Maximum distance:", max_dist)

## Get GraceDB superevents

In [ ]:
from desi_aap.gracedb_tools import (
    fetch_gracedb_superevents,
    run_3d_spatial_crossmatch,
    REQUIRE_2D_CREDIBLE_LEVEL,
)

In [ ]:
gracedb_events = fetch_gracedb_superevents(se_types=["BNS", "NSBH"])
gracedb_events

## Run temporal crossmatch between transients (SESN) and superevents (GW)

In [ ]:
from desi_aap.gracedb_tools import temporal_crossmatch_sesn_to_gw

In [ ]:
df_sesn_gracedb_temporal = temporal_crossmatch_sesn_to_gw(df_sesn, gracedb_events)
df_sesn_gracedb_temporal

In [ ]:
### TODO: modularize this too?

temporal_summary = (
    df_sesn_gracedb_temporal.groupby(["superevent_id", "gw_time"], dropna=False)
    .size()
    .rename("n_temporal_sesn")
    .reset_index()
    if not df_sesn_gracedb_temporal.empty
    else pd.DataFrame(columns=["superevent_id", "gw_time", "n_temporal_sesn"])
)

if not gracedb_events.empty:
    gracedb_temporal_summary = gracedb_events.merge(
        temporal_summary, on=["superevent_id", "gw_time"], how="left"
    )
    gracedb_temporal_summary["n_temporal_sesn"] = (
        gracedb_temporal_summary["n_temporal_sesn"].fillna(0).astype(int)
    )
else:
    gracedb_temporal_summary = pd.DataFrame()

display_cols = [
    "superevent_id",
    "gw_time",
    "far_per_year",
    "p_bns",
    "p_nsbh",
    "pipeline",
    "search",
    "skymap_file",
    "n_temporal_sesn",
    "status",
]
gracedb_temporal_summary[[c for c in display_cols if c in gracedb_temporal_summary.columns]]

**About 2D vs 3D Ranking**

`ligo.skymap.postprocess.crossmatch` reports two different rankings. `searched_prob_2d` is the sky-only credible level after marginalizing over distance. `searched_prob_vol` is a 3D voxel credible level ranked by posterior density per volume at `(RA, Dec, distance)`. These are not nested quantities, so `searched_prob_vol` can be much smaller than `searched_prob_2d` when the SN distance falls on a high-density distance slice even if the sky position is not in the highest-probability sky pixels.

In [ ]:
df_sesn_gracedb_3d = run_3d_spatial_crossmatch(df_sesn_gracedb_temporal, gracedb_events)
df_sesn_gracedb_3d

In [ ]:
### TODO: would like to modularize this one also maybe

if df_sesn_gracedb_3d.empty or not {"spatial_status", "inside_3d_credible_level"}.issubset(
    df_sesn_gracedb_3d.columns
):
    coincidence_sne = pd.DataFrame()
else:
    coincidence_mask = (df_sesn_gracedb_3d["spatial_status"] == "ok") & (
        df_sesn_gracedb_3d["inside_3d_credible_level"] == True
    )
    if REQUIRE_2D_CREDIBLE_LEVEL:
        coincidence_mask &= df_sesn_gracedb_3d["inside_2d_credible_level"] == True
    coincidence_sne = df_sesn_gracedb_3d[coincidence_mask].copy()

coincidence_display_cols = [
    "superevent_id",
    "gw_time",
    "gw_far_per_year",
    "gw_p_bns",
    "gw_p_nsbh",
    "name",
    "type",
    "discoverydate",
    "days_from_gw",
    "redshift",
    "cosmology",
    "sn_dist_mpc",
    "searched_prob_2d",
    "searched_prob_3d_density_rank",
    "searched_prob_dist",
    "searched_area_deg2",
    "credible_area_deg2",
    "credible_volume_mpc3",
    "inside_2d_credible_level",
    "inside_3d_credible_level",
    "ra",
    "declination",
    "reporting_group",
    "internal_names",
]
coincidence_sne[[c for c in coincidence_display_cols if c in coincidence_sne.columns]]

## Plot results

In [ ]:
from desi_aap.skymap_plots import plot_3d_coincidence

In [ ]:
# Try a single plot

plot_3d_coincidence(coincidence_sne.iloc[0], gracedb_events, show=True)

In [ ]:
plot_paths = []
if not coincidence_sne.empty:
    for i, row in coincidence_sne.iterrows():
        plot_paths.append(plot_3d_coincidence(row, gracedb_events))

pd.DataFrame({"plot_path": [str(path) for path in plot_paths]})